# CS-49 — Đánh giá crab detector

Tính precision / recall / mAP50 trên tập **test** giữ riêng và ghi báo cáo.

Quyết định image-level: ảnh có cua nếu có ít nhất 1 box vượt ngưỡng tin cậy.


In [ ]:
# @title Cài Ultralytics
!pip -q install ultralytics==8.3.70


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

WEIGHTS = Path('/content/crabshare-ml/crab-detector-v1/weights/best.pt')
DATA_YAML = Path('/content/drive/MyDrive/CrabShare-ML/splits/crab/data.yaml')
REPORT_DIR = Path('/content/drive/MyDrive/CrabShare-ML/report')
# REPORT_DIR = Path('ml/report')

CONF = 0.25
IOU = 0.5
REPORT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from ultralytics import YOLO

model = YOLO(str(WEIGHTS))
metrics = model.val(data=str(DATA_YAML), split='test', conf=CONF, iou=IOU, imgsz=640)
box = metrics.box
summary = {
    'model_version': 'crab-detector-v1',
    'weights': str(WEIGHTS),
    'conf': CONF,
    'iou': IOU,
    'precision': float(box.mp),
    'recall': float(box.mr),
    'mAP50': float(box.map50),
    'mAP50_95': float(box.map),
    'evaluated_at': datetime.now(timezone.utc).isoformat(),
}
print(json.dumps(summary, indent=2))
(REPORT_DIR / 'metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Wrote', REPORT_DIR / 'metrics.json')

ok = summary['precision'] >= 0.85 and summary['recall'] >= 0.80
print('PASS targets' if ok else 'BELOW targets — giữ tín hiệu yếu trong risk_score')


In [ ]:
notes = '''# Misdetections

## False positives
- (điền sau khi xem runs/val)

## False negatives
- (điền sau khi xem runs/val)

## Ảnh cần bổ sung
- (ví dụ: nước đục, nắp nửa mở, cua núp góc)
'''
(REPORT_DIR / 'misdetections.md').write_text(notes, encoding='utf-8')
print('Wrote misdetections.md')
